# BF16 Special Panels: Surface · IsoFLOP

Variant of the main panel notebook (v16) for the BF16 sweep.
Convergence panel is **omitted** (no seed-variance run was done
for BF16). Surface and IsoFLOP panels inherit every styling
improvement from v16, so the two figure sets stay visually
consistent.


In [ ]:
# ============================================================
# 0 · SETUP - BF16 special-results variant
# ============================================================
import sys, os, glob, importlib

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import to_rgba
from mpl_toolkits.mplot3d import Axes3D
from scipy.optimize import minimize_scalar
from copy import deepcopy
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.family']      = 'STIXGeneral'

# ── Paths ──
# Notebook lives at  <repo>/src/tpp_analysis/visualizations/
# BF16 parquets at   <repo>/special_results/
# CSVs at            <repo>/extracted_losses/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
DATA_DIR     = os.path.join(PROJECT_ROOT, 'extracted_losses')
SAVE_DIR     = os.path.join(PROJECT_ROOT, 'special_results')
FIG_DIR      = os.path.join(SAVE_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

# Convergence panel intentionally disabled for BF16 - no seed-variance
# results exist for this sweep. Set SV_SAVE_DIR to a path that won\'t
# be found, so any accidental sv lookup returns False.
SV_SAVE_DIR  = os.path.join(SAVE_DIR, '_no_sv_for_bf16')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
_TPP = os.path.join(PROJECT_ROOT, 'src', 'tpp_analysis')
if _TPP not in sys.path:
    sys.path.insert(0, _TPP)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA_DIR:     {DATA_DIR}')
print(f'SAVE_DIR:     {SAVE_DIR}')
print(f'FIG_DIR:      {FIG_DIR}')

for p in [DATA_DIR, SAVE_DIR]:
    print(f'  {"✓" if os.path.isdir(p) else "✗"}  {p}')


In [ ]:
# ============================================================
# 1 · IMPORTS FROM PROJECT MODULES
# ============================================================
from scaling_laws import run_full_analysis, load_and_prepare
from scaling_laws.data import prepare_arrays, PreparedData
from scaling_laws.laws import LAWS

import incremental_coverage.viz as _viz_mod
importlib.reload(_viz_mod)
from incremental_coverage.viz import plot_single_comparison, DESIGN_COLORS

from seed_variance import plot_convergence_with_errorbars

print('Imports OK')


In [ ]:
# ============================================================
# 2 · LOAD DATA - BF16 special-results variant
# ============================================================

def load_pipeline_from_parquet(parquet_path):
    """Same loader as the main notebook, with one extra dataset name
    map entry for BF16 ('Wikipedia_BF16' → 'Wikipedia BF16')."""
    df = pd.read_parquet(parquet_path)

    PARAM_ORDER = {
        'Chinchilla':    ['param_0', 'param_1', 'param_2', 'param_3', 'param_4'],
        'Kaplan':        ['param_0', 'param_1', 'param_2', 'param_3'],
        'Droppo-Elibol': ['param_0', 'param_1', 'param_2', 'param_3', 'param_4', 'param_5'],
        'Parametric-Epoch': ['param_A', 'param_alpha', 'param_B', 'param_beta',
                             'param_E', 'param_R_star_D', 'param_R_star_N'],
    }

    DESIGN_MAP  = {'CO': 'Colinear', 'NC': 'Non-Colinear'}
    DS_NAME_MAP = {
        'RedPajama': 'Red Pajama',
        'Wikipedia_BF16': 'Wikipedia BF16',
    }

    results = []
    seed0 = df[df['seed'] == 0]

    for (ds, law, mode, design), row in seed0.groupby(['dataset', 'law', 'mode', 'design']):
        row = row.iloc[0]
        param_cols = PARAM_ORDER.get(law, [])
        params = [float(row[c]) for c in param_cols if c in row.index and pd.notna(row[c])]

        results.append({
            'dataset': DS_NAME_MAP.get(ds, ds),
            'law': law,
            'mode': mode.capitalize() if mode != 'Epoch-Expanded' else 'Epoch-Expanded',
            'design': DESIGN_MAP.get(design, design),
            'params': params,
            'r2_train': row.get('r2_train', np.nan),
            'r2_unified': row.get('r2_unified', np.nan),
            'r2_colinear': row.get('r2_colinear', np.nan),
            'r2_noncolinear': row.get('r2_noncolinear', np.nan),
            'rmse_train': row.get('rmse_train', np.nan),
            'rmse_unified': row.get('rmse_unified', np.nan),
            'rmse_colinear': row.get('rmse_colinear', np.nan),
            'rmse_noncolinear': row.get('rmse_noncolinear', np.nan),
        })

    return results


# ── Convergence panel disabled for BF16 ──
# No seed-variance and no incremental-coverage results were produced for
# this sweep. draw_panels() will still receive these objects but is called
# with skip_convergence=True (see batch cell below).
inc_results = {}
sv_results  = {}
print('BF16 variant: convergence panel disabled (no seed-variance data)')


# ── Pipeline results from BF16 master parquet ──
master_parquet = os.path.join(SAVE_DIR, 'full_coverage_bf16.parquet')
if os.path.exists(master_parquet):
    pipeline_results = load_pipeline_from_parquet(master_parquet)
    print(f'\nPipeline results: {len(pipeline_results)} entries (from {os.path.basename(master_parquet)})')
    combos = sorted({(r["dataset"], r["law"], r["mode"]) for r in pipeline_results})
    print(f'\n(dataset, law, mode) combos found:')
    for ds, law, mode in combos:
        n = sum(1 for r in pipeline_results
                if r["dataset"] == ds and r["law"] == law and r["mode"] == mode)
        print(f'  {ds:20s} | {law:16s} | {mode:14s}  ({n} designs)')
else:
    pipeline_results = None
    print(f'⚠️  {master_parquet} not found')


# ── PreparedData ──
# CSVs live in DATA_DIR (extracted_losses/). Try the standard name
# variants in priority order; override BF16_CSV_PATH manually if needed.
BF16_DATASET_LABEL = 'Wikipedia BF16'

_candidates = [
    os.path.join(DATA_DIR, 'wikipedia_bf16.csv'),
    os.path.join(DATA_DIR, 'Wikipedia_BF16.csv'),
    os.path.join(DATA_DIR, 'wikipedia-bf16.csv'),
    os.path.join(SAVE_DIR, 'wikipedia_bf16.csv'),
    os.path.join(SAVE_DIR, 'Wikipedia_BF16.csv'),
]
BF16_CSV_PATH = next((p for p in _candidates if os.path.exists(p)), None)

prepared_data = {}
if BF16_CSV_PATH is not None:
    prepared_data[BF16_DATASET_LABEL] = load_and_prepare(
        BF16_CSV_PATH, BF16_DATASET_LABEL, verbose=False)
    print(f'\n✓ Loaded BF16 CSV: {BF16_CSV_PATH}')
else:
    print('\n⚠️  No BF16 CSV found in any standard location:')
    for p in _candidates:
        print(f'    {p}')
    print('   Set BF16_CSV_PATH manually and re-run this cell.')

print(f'\nPreparedData keys: {list(prepared_data.keys())}')


---
## 3 · Shared Sizing Constants

Everything inherits from the surface-plot baseline so the panels
look uniform at any scale-down.


In [ ]:
# ============================================================
# 3 · GLOBAL SIZING - surface plot is the baseline
# ============================================================
# All sizing/style constants and helpers (including _sv_tag) are
# imported from panel_style.
_VIZ_DIR = os.path.dirname(os.path.abspath('__file__'))
if _VIZ_DIR not in sys.path:
    sys.path.insert(0, _VIZ_DIR)
from panel_style import (
    SURF_W, SURF_H, PANEL_W, PANEL_H,
    AXIS_LABEL_FS, TICK_LABEL_FS, LEGEND_FS, TITLE_FS,
    TICK_MAJOR_LEN, TICK_MAJOR_WID, TICK_MINOR_LEN, TICK_MINOR_WID,
    LINE_WIDTH, MARKER_SIZE, SPINE_LW,
    INTERSECTION_LW, INTERSECTION_LS, INTERSECTION_ALPHA,
    GRID_BG_COLOR, LEGEND_ALPHA,
    GRID_LINE_COLOR, GRID_LINE_WIDTH, GRID_LINE_ALPHA,
    DPI,
    _sv_tag,
    FixedOrderFormatter,
    style_ticks_2d, style_ticks_3d,
    _style_legend, _style_all_legends,
    _darken_errorbars,
)
print('Sizing constants set.')

---
## 4 · Loss Functions, IsoFLOP Helpers, & Surface Helpers

In [ ]:
# ============================================================
# 4 · LOSS FUNCTIONS + ISOFLOP HELPERS
# ============================================================
# All loss functions, isoFLOP helpers and the _draw_isoflop_generic
# renderer live in the shared module isoflop_panel.py so the same
# code drives this notebook, the sibling notebook and figure1_panels.py.
_VIZ_DIR = os.path.dirname(os.path.abspath('__file__'))
if _VIZ_DIR not in sys.path:
    sys.path.insert(0, _VIZ_DIR)
from isoflop_panel import (
    LAW_LOSS_FNS,
    ISOFLOP_COLORS,
    generic_isoflop,
    generic_optimal_N,
    draw_isoflop_panel as _draw_isoflop_generic,
)
print('Loss functions + isoFLOP helpers + _draw_isoflop_generic imported.')


In [ ]:
# ============================================================
# 5 · SURFACE-PLOT HELPERS  (imported from surface_helpers)
# ============================================================
from surface_helpers import (
    _HOLDOUT_STYLES,
    _holdout_style,
    _r2_key_for_holdout,
    _build_adaptive_rgba,
    _camera_depth,
    _draw_boundary_edges_occlusion,
    _compute_intersection_3d,
)
print('Surface-plot helpers imported.')

In [ ]:
# ============================================================
# 6 · ISOFLOP BUDGET HELPER  (imported from isoflop_panel)
# ============================================================
from isoflop_panel import compute_flop_budgets
print('compute_flop_budgets imported.')

---
## 7 · Tick-Styling Helpers

In [ ]:
# ============================================================
# 7 · TICK / LEGEND / ERRORBAR STYLING  (imported from panel_style)
# ============================================================
print('Tick + legend + errorbar styling helpers imported.')

---
## 8 · Pick Your Combo & Build Individual Panels


In [ ]:
# ============================================================
# 11 · BATCH - generate individual panels for every top-N candidate
# ============================================================
# Requires `clean` ranking DataFrame from the viz notebook.

from matplotlib.ticker import ScalarFormatter, MultipleLocator

# Extra padding around 3-D plots so rotated z-labels don't get cropped
_SAVE_PAD_3D = 0.4
_SAVE_PAD_2D = 0.1


def draw_panels(
    dataset_name, law_name, mode_name,
    pipeline_results, prepared_data, inc_results,
    # ── shared sizing (uses globals) ──
    holdout_col='r2_unified', holdout_key='Unified',
    holdout_type='unified', min_coverage=0.15,
    # ── surface ──
    side_elev=15, side_azim=-35,
    co_color='steelblue', nc_color='indianred',
    # ── isoflop ──
    n_range=None, y_lim_iso=None,
    show_optimal=False, n_isoflop_curves=5,
    # ── convergence ──
    # y_min_conv=None  →  auto-fit just below the lowest real point at coverage ≥ 0.2
    y_min_conv=None, y_max_conv=1.02,
    # ── output ──
    save_dir=None, file_prefix='', dpi=300,
    # ── BF16 variant ──
    skip_convergence=False,
):
    """
    Draw and save individual panel figures (surface, convergence, isoflop).
    Returns a dict of Figures keyed by panel name.
    """
    if n_range is None:
        n_range = np.logspace(np.log10(2e7), np.log10(2e8), 500)

    matches = [r for r in pipeline_results
               if r['law'] == law_name
               and r['mode'] == mode_name
               and r['dataset'] == dataset_name]
    co_result = [r for r in matches if r['design'] == 'Colinear']
    nc_result = [r for r in matches if r['design'] == 'Non-Colinear']
    if not co_result or not nc_result:
        print(f'  ⚠️  No CO+NC pair for {dataset_name}/{law_name}/{mode_name}')
        return None
    co_result = co_result[0]
    nc_result = nc_result[0]

    law_obj = next((l for l in LAWS if l.name == law_name), None)
    if law_obj is None:
        print(f'  ⚠️  Unknown law: {law_name}')
        return None

    pdata = prepared_data.get(dataset_name)
    if pdata is None:
        print(f'  ⚠️  No PreparedData for {dataset_name}')
        return None

    inc_df = inc_results.get(dataset_name)
    has_convergence = inc_df is not None

    holdout_map = {
        'Unified':    pdata.unified_holdout,
        'CO Holdout': pdata.colinear_holdout,
        'NC Holdout': pdata.noncolinear_holdout,
    }
    holdouts = {holdout_key: holdout_map.get(holdout_key, pdata.unified_holdout)}

    flop_budgets = compute_flop_budgets(
        pdata, holdout_type=holdout_type, n_curves=n_isoflop_curves)

    _r2_key = _r2_key_for_holdout(holdout_key)
    co_r2 = co_result.get(_r2_key, co_result.get('r2_train', float('nan')))
    nc_r2 = nc_result.get(_r2_key, nc_result.get('r2_train', float('nan')))

    co_params = co_result['params']
    nc_params = nc_result['params']

    ds_tag   = dataset_name.replace(' ', '_')
    law_tag  = law_name.replace(' ', '_').replace('-', '_')
    mode_tag = mode_name.replace(' ', '_')
    base_tag = f'{ds_tag}_{law_tag}_{mode_tag}'

    figs = {}

    # ══════════════════════════════════════════════════════════
    # SURFACE PLOT (3-D)
    # ══════════════════════════════════════════════════════════
    fig_surf = plt.figure(figsize=(SURF_W, SURF_H))
    ax_surf = fig_surf.add_subplot(111, projection='3d')

    all_N, all_D = np.array([]), np.array([])
    prepped = {}
    for h_label, h_list in holdouts.items():
        if not h_list: continue
        Nh, Dh, Lh = prepare_arrays(h_list, 'final')
        if len(Lh) > 0:
            prepped[h_label] = (Nh, Dh, Lh)
            all_N = np.concatenate([all_N, Nh])
            all_D = np.concatenate([all_D, Dh])

    if len(all_N) == 0:
        print(f'  ⚠️  No holdout data for {dataset_name}')
        plt.close(fig_surf)
        return None

    N_1d = np.logspace(np.log10(all_N.min()), np.log10(all_N.max()), 200)
    D_1d = np.logspace(np.log10(all_D.min()), np.log10(all_D.max()), 200)
    N_mesh, D_mesh = np.meshgrid(N_1d, D_1d)

    L_co = law_obj.loss_fn(N_mesh, D_mesh, *co_params)
    L_nc = law_obj.loss_fn(N_mesh, D_mesh, *nc_params)

    all_L = np.concatenate([v[2] for v in prepped.values()])
    L_lo  = np.nanpercentile(all_L, 1) * 0.5
    L_hi  = np.nanpercentile(all_L, 99) * 1.5
    L_co_clip = np.clip(L_co, L_lo, L_hi)
    L_nc_clip = np.clip(L_nc, L_lo, L_hi)
    z_lo = min(L_lo, L_co_clip.min(), L_nc_clip.min()) * 0.95
    z_hi = max(L_hi, L_co_clip.max(), L_nc_clip.max()) * 1.05
    n_lim = (N_mesh.min(), N_mesh.max())
    d_lim = (D_mesh.min(), D_mesh.max())
    z_lim = (z_lo, z_hi)

    co_rgba = _build_adaptive_rgba(co_color, L_co_clip, L_nc_clip)
    nc_rgba = _build_adaptive_rgba(nc_color, L_nc_clip, L_co_clip)

    ax_surf.plot_surface(N_mesh, D_mesh, L_co_clip,
                         facecolors=co_rgba, edgecolor='none',
                         rstride=2, cstride=2, shade=False, zorder=1)
    ax_surf.plot_surface(N_mesh, D_mesh, L_nc_clip,
                         facecolors=nc_rgba, edgecolor='none',
                         rstride=2, cstride=2, shade=False, zorder=2)

    _draw_boundary_edges_occlusion(ax_surf, N_mesh, D_mesh,
                                    L_co_clip, L_nc_clip, co_color,
                                    side_elev, side_azim, n_lim, d_lim, z_lim)
    _draw_boundary_edges_occlusion(ax_surf, N_mesh, D_mesh,
                                    L_nc_clip, L_co_clip, nc_color,
                                    side_elev, side_azim, n_lim, d_lim, z_lim)

    ax_surf.plot([], [], [], color=co_color, lw=6, alpha=0.8,
                 label=f'CO (R²={co_r2:.3f})')
    ax_surf.plot([], [], [], color=nc_color, lw=6, alpha=0.8,
                 label=f'NC (R²={nc_r2:.3f})')

    for h_label, (Nh, Dh, Lh) in prepped.items():
        sty = _holdout_style(h_label)
        ax_surf.scatter(Nh, Dh, Lh, c=sty['color'], s=sty['s'], alpha=0.85,
                        edgecolors='white', linewidths=0.5,
                        marker=sty['marker'], label=h_label.replace('Unified', 'Holdout').replace('unified', 'Holdout'), zorder=5)

    bar_key = next(iter(prepped))
    N_b, D_b, L_b = prepped[bar_key]
    L_pred_co = law_obj.loss_fn(N_b, D_b, *co_params)
    L_pred_nc = law_obj.loss_fn(N_b, D_b, *nc_params)
    tick_len  = (all_N.max() - all_N.min()) * 0.008
    drawn_co = drawn_nc = False
    for i in range(len(N_b)):
        ni, di, li = N_b[i], D_b[i], L_b[i]
        ax_surf.plot([ni,ni],[di,di],[li, L_pred_co[i]],
                    color='green', lw=1.8, alpha=0.8, zorder=4,
                    label='CO residual' if not drawn_co else None)
        ax_surf.plot([ni-tick_len,ni+tick_len],[di,di],
                    [L_pred_co[i],L_pred_co[i]],
                    color='green', lw=1.0, alpha=0.8, zorder=4)
        drawn_co = True
        ax_surf.plot([ni,ni],[di,di],[li, L_pred_nc[i]],
                    color='black', lw=1.8, alpha=0.8, zorder=4,
                    label='NC residual' if not drawn_nc else None)
        ax_surf.plot([ni-tick_len,ni+tick_len],[di,di],
                    [L_pred_nc[i],L_pred_nc[i]],
                    color='black', lw=1.0, alpha=0.8, zorder=4)
        drawn_nc = True

    # Plain 'N' / 'D' - multiplier rendered separately by the formatter.
    # labelpad bumped 10 → 18 so labels sit clear of tick numbers.
    ax_surf.set_xlabel('N', fontsize=AXIS_LABEL_FS, labelpad=18)
    ax_surf.set_ylabel('D', fontsize=AXIS_LABEL_FS, labelpad=18)
    # Smaller labelpad so rotated "Loss" doesn't get cropped by savefig
    # 'Loss' via text2D in axes-fraction coords - see cell 15 note.
    ax_surf.set_zlabel('')
    ax_surf.text2D(1.08, 0.5, 'Loss',
                   transform=ax_surf.transAxes,
                   rotation=90, va='center', ha='left',
                   fontsize=AXIS_LABEL_FS)

    # N axis: standard auto-offset (gives "1e7" with integer-ish ticks).
    _sf_n = ScalarFormatter(useMathText=False)
    _sf_n.set_scientific(True)
    _sf_n.set_powerlimits((0, 0))
    ax_surf.xaxis.set_major_formatter(_sf_n)

    # D axis: ticks 0, 2, ..., 14 with locked '1e8' offset, mirroring N.
    ax_surf.yaxis.set_major_locator(MultipleLocator(2e8))
    ax_surf.yaxis.set_major_formatter(FixedOrderFormatter(oom=8, fmt='%d'))

    ax_surf.invert_yaxis()
    ax_surf.set_zlim(z_lo, z_hi)
    ax_surf.view_init(elev=side_elev, azim=side_azim)
    ax_surf.legend(fontsize=LEGEND_FS, loc='upper left', framealpha=LEGEND_ALPHA)
    style_ticks_3d(ax_surf)
    _style_all_legends(ax_surf)

    if save_dir:
        for ext in ['pdf', 'png']:
            out = os.path.join(save_dir, f'{file_prefix}surface_{base_tag}.{ext}')
            fig_surf.savefig(out, dpi=dpi, bbox_inches='tight',
                             pad_inches=_SAVE_PAD_3D)
            print(f'  Saved: {out}')
    plt.show()
    plt.close(fig_surf)
    figs['surface'] = fig_surf

    # ══════════════════════════════════════════════════════════
    # CONVERGENCE
    # ══════════════════════════════════════════════════════════
    sv_tag = _sv_tag(dataset_name, law_name, mode_name)
    sv_path = os.path.join(SV_SAVE_DIR, f"seed_variance_{sv_tag}.parquet")
    has_sv = os.path.exists(sv_path)

    def _auto_ymin(df, col, anchor_cov=0.20, margin=0.05, floor=-0.5):
        if df is None or df.empty or 'coverage_frac' not in df.columns or col is None:
            return -0.1
        sub = df.loc[df['coverage_frac'] >= anchor_cov, col].dropna()
        if sub.empty:
            return -0.1
        return max(float(sub.min()) - margin, floor)

    if not skip_convergence and (has_sv or has_convergence):
        fig_conv = plt.figure(figsize=(PANEL_W, PANEL_H))
        ax_conv = fig_conv.add_subplot(111)

        if has_sv:
            sv_df = pd.read_parquet(sv_path)
            r2_col = next(
                (c for c in ('r2_unified', 'r2_holdout_unified', 'r2_unified_holdout')
                 if c in sv_df.columns),
                None,
            )
            if r2_col is None:
                cands = [c for c in sv_df.columns if 'r2' in c.lower()]
                r2_col = cands[0] if cands else None

            y_min_use = (_auto_ymin(sv_df, r2_col)
                         if y_min_conv is None else y_min_conv)

            plot_convergence_with_errorbars(
                sv_df,
                holdout_split='unified',
                ax=ax_conv,
                y_min=y_min_use,
                y_max=y_max_conv,
                min_coverage=min_coverage,
                band_type='percentile',
                axis_label_fs=AXIS_LABEL_FS,
                tick_label_fs=TICK_LABEL_FS,
                legend_fs=LEGEND_FS,
                marker_size=MARKER_SIZE,
                line_width=LINE_WIDTH,
            )
            leg = ax_conv.get_legend()
            if leg is not None:
                for txt in leg.get_texts():
                    new = txt.get_text().replace('Non-Colinear', 'NC').replace('Colinear', 'CO')
                    txt.set_text(new)
            ax_conv.set_ylabel(r'Holdout $R^2$', fontsize=AXIS_LABEL_FS)
            print(f'  Panel B: seed-variance (tag={sv_tag}, y_min={y_min_use:.3f})')
        elif has_convergence:
            conv_sub = inc_df[
                (inc_df['law'] == law_name)
                & (inc_df['mode'] == mode_name)
                & inc_df['success']
            ].copy()
            if min_coverage is not None:
                conv_sub = conv_sub[conv_sub['coverage_frac'] >= min_coverage]
            _ABBREV = {'Colinear': 'CO', 'Non-Colinear': 'NC'}

            y_min_use = (_auto_ymin(conv_sub, holdout_col)
                         if y_min_conv is None else y_min_conv)

            for design in ['Colinear', 'Non-Colinear']:
                dsub = conv_sub[conv_sub['design'] == design].sort_values('coverage_frac')
                if dsub.empty: continue
                line_color = co_color if design == 'Colinear' else nc_color
                ax_conv.plot(
                    dsub['coverage_frac'], dsub[holdout_col],
                    color=line_color,
                    marker='o' if design == 'Colinear' else 's',
                    markersize=MARKER_SIZE, linewidth=LINE_WIDTH,
                    label=_ABBREV[design],
                )
            ax_conv.set_xlabel('Coverage Fraction', fontsize=AXIS_LABEL_FS)
            ax_conv.set_ylabel(r'Holdout $R^2$',    fontsize=AXIS_LABEL_FS)
            ax_conv.set_xlim(-0.02, 1.05)
            ax_conv.set_ylim(y_min_use, y_max_conv)
            ax_conv.legend(fontsize=LEGEND_FS, loc='lower right',
                           framealpha=LEGEND_ALPHA)
            ax_conv.grid(True, alpha=0.3)
            print(f'  Panel B: single-seed fallback (tag={sv_tag} not found, y_min={y_min_use:.3f})')

        style_ticks_2d(ax_conv)
        # Darken seed_variance's errorbars to pop on the GRID_BG_COLOR bg
        _darken_errorbars(ax_conv)
        _style_all_legends(ax_conv)

        if save_dir:
            for ext in ['pdf', 'png']:
                out = os.path.join(save_dir, f'{file_prefix}convergence_{base_tag}.{ext}')
                fig_conv.savefig(out, dpi=dpi, bbox_inches='tight',
                                 pad_inches=_SAVE_PAD_2D)
                print(f'  Saved: {out}')
        plt.show()
        plt.close(fig_conv)
        figs['convergence'] = fig_conv

    # ══════════════════════════════════════════════════════════
    # ISOFLOP CURVES
    # ══════════════════════════════════════════════════════════
    fig_iso = plt.figure(figsize=(PANEL_W, PANEL_H))
    ax_iso = fig_iso.add_subplot(111)

    iso_fits = [r for r in pipeline_results
                if r['law'] == law_name
                and r['mode'] == mode_name
                and r['dataset'] == dataset_name]

    if holdout_type == 'unified':
        h_thin = pdata.unified_holdout
    elif holdout_type == 'colinear':
        h_thin = pdata.colinear_holdout
    else:
        h_thin = pdata.noncolinear_holdout

    _draw_isoflop_generic(
        ax_iso, iso_fits, law_name, h_thin,
        'Holdout',
        flop_budgets, n_range, y_lim_iso,
        show_optimal=show_optimal,
        fontsize=AXIS_LABEL_FS,
        legend_fontsize=20,
        legend_loc='lower left',
        line_width=LINE_WIDTH,
        holdout_mode=mode_name.lower(),
        style_legends_fn=_style_all_legends,
    )
    style_ticks_2d(ax_iso)
    _style_all_legends(ax_iso)

    if save_dir:
        for ext in ['pdf', 'png']:
            out = os.path.join(save_dir, f'{file_prefix}isoflop_{base_tag}.{ext}')
            fig_iso.savefig(out, dpi=dpi, bbox_inches='tight',
                            pad_inches=_SAVE_PAD_2D)
            print(f'  Saved: {out}')
    plt.show()
    plt.close(fig_iso)
    figs['isoflop'] = fig_iso

    return figs


print('draw_panels() defined.')


---
## BF16 Batch Run
Runs surface + isoFLOP panels for every (law, mode) combo found
in the BF16 parquet. Convergence panel is skipped via
`skip_convergence=True`. Same dimensions and styling as the main
notebook so figures stay visually consistent.


In [ ]:
# ============================================================
# 12 · BF16 BATCH - surface + isoFLOP for every (law, mode) combo
# ============================================================

# Mirror main-notebook draw_panels kwargs for visual parity
HOLDOUT_COL  = 'r2_unified'
HOLDOUT_KEY  = 'Unified'
HOLDOUT_TYPE = 'unified'
MIN_COVERAGE = 0.15

SIDE_ELEV    = 15
SIDE_AZIM    = -35
CO_COLOR     = 'steelblue'
NC_COLOR     = 'indianred'

N_RANGE          = np.logspace(np.log10(2e7), np.log10(2e8), 500)
Y_LIM_ISO        = None
SHOW_OPTIMAL     = False
N_ISOFLOP_CURVES = 5

# Convergence params still passed (signature compat) but ignored due to
# skip_convergence=True below.
Y_MIN_CONV   = -0.1
Y_MAX_CONV   = 1.02

# ── Discover combos in pipeline_results that we have CSVs for ──
assert pipeline_results is not None, 'pipeline_results not loaded'
assert prepared_data, 'No prepared_data - set BF16_CSV_PATH and re-run cell 3'

bf16_combos = sorted({
    (r['dataset'], r['law'], r['mode'])
    for r in pipeline_results
    if r['dataset'] in prepared_data
})

print(f'BF16 combos to render: {len(bf16_combos)}')
for ds, law, mode in bf16_combos:
    print(f'  {ds:20s} | {law:16s} | {mode}')

# ── Run ──
for rank, (ds, law, mode) in enumerate(bf16_combos, 1):
    print(f"\n{'#'*70}")
    print(f'  BF16 #{rank}: {ds} / {law} / {mode}')
    print(f"{'#'*70}")

    try:
        draw_panels(
            dataset_name=ds,
            law_name=law,
            mode_name=mode,
            pipeline_results=pipeline_results,
            prepared_data=prepared_data,
            inc_results=inc_results,
            holdout_col=HOLDOUT_COL,
            holdout_key=HOLDOUT_KEY,
            holdout_type=HOLDOUT_TYPE,
            min_coverage=MIN_COVERAGE,
            side_elev=SIDE_ELEV,
            side_azim=SIDE_AZIM,
            co_color=CO_COLOR,
            nc_color=NC_COLOR,
            n_range=N_RANGE,
            y_lim_iso=Y_LIM_ISO,
            show_optimal=SHOW_OPTIMAL,
            n_isoflop_curves=N_ISOFLOP_CURVES,
            y_min_conv=Y_MIN_CONV,
            y_max_conv=Y_MAX_CONV,
            save_dir=FIG_DIR,
            file_prefix=f'bf16_{rank:02d}_',
            dpi=DPI,
            # ── BF16-specific: don\'t draw the convergence panel ──
            skip_convergence=True,
        )
    except Exception as e:
        print(f'  ⚠️  Failed: {type(e).__name__}: {e}')
        import traceback; traceback.print_exc()

print(f'\nDone - {len(bf16_combos)} BF16 panel sets generated in {FIG_DIR}.')
